# Finance Q&A Agent: verified inputs, sandboxed code, measured accuracy

Answers business and finance questions (growth, churn, margins, break-even) for product managers. The LLM **plans** the calculation; **code** checks the inputs and does the arithmetic.

| Feature | What it does |
|---|---|
| **Every number traced to the question** | The model must list each input with the exact phrase it came from. Code checks that the phrase is in the question and that the value matches it (so "8%" may become 0.08 but not 0.8), and the calculation code may not contain any other hard-coded numbers |
| **Ask instead of guess** | If a question is missing data or is ambiguous in a way that changes the answer, the agent asks a clarifying question or states its assumption, instead of producing a confident number |
| **Measured against a baseline** | 18 labelled questions are answered two ways: the LLM answering directly vs. this agent. A scorecard compares accuracy, clarifications and failures |

Generated code runs in a **restricted sandbox**: no imports, attribute access or unknown functions, and the printed result is captured.

**Stack:** OpenAI or Anthropic via LangChain · Pydantic for the structured reply · no Azure needed.

**Setup:** add `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` to Colab **Secrets** (🔑) with notebook access on, and set `PROVIDER` in Step 5. The offline tests in Step 4 need no key.

## Step 1: Install packages

In [ ]:
!pip install -q -U langchain-core langchain-openai langchain-anthropic pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.0/133.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.8/573.8 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 488.1/488.1 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 39.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 862.5/862.5 kB 40.0 MB/s eta 0:00:00


## Step 2: The structured reply and the input checks

The model replies in JSON with either:

- `"status": "answer"` with its **inputs** (name, value, source phrase), **assumptions**, **code** and an explanation; or
- `"status": "clarify"` with a clarifying question, when an answer would mean guessing.

`check_inputs` then verifies every input against the question.

In [ ]:
import json
import re
from typing import Literal

from pydantic import BaseModel, Field, ValidationError


class Input(BaseModel):
  name: str = Field(pattern=r"^[a-z_][a-z0-9_]*$")
  value: float
  source: str = Field(min_length=1)          # the exact phrase from the question


class AgentReply(BaseModel):
  status: Literal["answer", "clarify"]
  clarifying_question: str | None = None
  assumptions: list[str] = []
  inputs: list[Input] = []
  code: str = ""
  explanation: str = ""


WORD_NUMBERS = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12, "double": 2, "half": 0.5}
NUMBER_RE = re.compile(r"(\d[\d,]*\.?\d*)\s*(%|(?:k|m|bn|b|thousand|million|billion|percent)\b)?", re.IGNORECASE)
SCALE = {"k": 1e3, "thousand": 1e3, "m": 1e6, "million": 1e6, "b": 1e9, "bn": 1e9, "billion": 1e9}


def normalise(text: str) -> str:
  text = text.lower().replace("’", "'").replace("“", '"').replace("”", '"')
  return re.sub(r"\s+", " ", text).strip()


def numbers_in(text: str) -> set:
  """Every value a phrase can legitimately stand for: '8%' -> {8, 0.08}, '$1.2M' -> {1.2, 1200000}."""

  values = set()
  for raw, suffix in NUMBER_RE.findall(text.replace("$", "")):
    n = float(raw.replace(",", ""))
    values.add(n)
    s = (suffix or "").lower()
    if s in SCALE:
      values.add(n * SCALE[s])
    if s in ("%", "percent"):
      values.add(n / 100)
  for word, n in WORD_NUMBERS.items():
    if re.search(rf"\b{word}\b", text.lower()):
      values.add(float(n))
  return values


def check_inputs(inputs: list, question: str) -> list:
  """Each input's source phrase must appear in the question, and its value must match a number in that phrase."""

  problems, q = [], normalise(question)
  for inp in inputs:
    if normalise(inp.source) not in q:
      problems.append(f"input '{inp.name}': source phrase \"{inp.source}\" does not appear in the question")
      continue
    if not any(abs(inp.value - v) <= 1e-9 * max(1, abs(v)) for v in numbers_in(inp.source)):
      problems.append(f"input '{inp.name}' = {inp.value} does not match any number in \"{inp.source}\"")
  return problems


def parse_reply(text: str) -> AgentReply:
  match = re.search(r"\{.*\}", text, re.DOTALL)
  if not match:
    raise ValueError("no JSON object found in the reply")
  return AgentReply.model_validate_json(match.group(0))

## Step 3: The sandbox: inputs are injected, nothing else is hard-coded

The verified inputs are passed into the sandbox as variables. The code itself may only use those variables, a few structural constants (`0, 1, 2, 12, 100`) and safe built-ins. A hard-coded `10000` in the code is rejected, because it would bypass the input check.

In [ ]:
import ast
import contextlib
import io

SAFE_BUILTINS = {"print": print, "round": round, "abs": abs, "min": min, "max": max, "sum": sum, "range": range, "len": len, "int": int, "float": float, "pow": pow}
ALLOWED_CONSTANTS = {0, 1, 2, 12, 100}
BLOCKED_NODES = (ast.Import, ast.ImportFrom, ast.With, ast.While, ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Lambda, ast.Attribute, ast.Global, ast.Nonlocal, ast.Try, ast.Raise, ast.Delete)


def validate_code(code: str, input_names: set) -> list:
    try:
        tree = ast.parse(code)
    except SyntaxError as err:
        return [f"syntax error: {err.msg} (line {err.lineno})"]
    assigned = {n.id for n in ast.walk(tree) if isinstance(n, ast.Name) and isinstance(n.ctx, ast.Store)}
    known = input_names | assigned | set(SAFE_BUILTINS)
    problems = []
    for node in ast.walk(tree):
        line = getattr(node, "lineno", "?")
        if isinstance(node, BLOCKED_NODES):
            problems.append(f"not allowed: {type(node).__name__} (line {line})")
        elif isinstance(node, ast.Call) and not (isinstance(node.func, ast.Name) and node.func.id in SAFE_BUILTINS):
            problems.append(f"not allowed: a call to something other than {sorted(SAFE_BUILTINS)} (line {line})")
        elif isinstance(node, ast.Name) and isinstance(node.ctx, ast.Load) and node.id not in known:
            problems.append(f"unknown variable '{node.id}' (line {line}): declare it as an input")
        elif isinstance(node, ast.Constant) and isinstance(node.value, (int, float)) and not isinstance(node.value, bool) \
                and node.value not in ALLOWED_CONSTANTS:
            problems.append(f"hard-coded number {node.value} (line {line}): declare it as an input with its source")
    if not any(isinstance(n, ast.Call) and getattr(n.func, "id", "") == "print" for n in ast.walk(tree)):
        problems.append("the code must print its answer with print(...)")
    return problems


def run_sandboxed(code: str, inputs: dict) -> dict:
    problems = validate_code(code, set(inputs))
    if problems:
        return {"ok": False, "output": "", "error": "; ".join(problems)}
    buffer = io.StringIO()
    try:
        with contextlib.redirect_stdout(buffer):
            exec(compile(code, "<generated>", "exec"), {"__builtins__": SAFE_BUILTINS, **inputs})   # one namespace, so generators see the inputs
        return {"ok": True, "output": buffer.getvalue().strip(), "error": ""}
    except Exception as err:
        return {"ok": False, "output": buffer.getvalue().strip(), "error": f"{type(err).__name__}: {err}"}


def first_number(text: str):
    match = re.search(r"-?\d[\d,]*\.?\d*", (text or "").replace("$", ""))
    return float(match.group(0).replace(",", "")) if match else None


def text_of(message) -> str:
    """OpenAI returns a string; Anthropic can return a list of content blocks."""
    if isinstance(message.content, str):
        return message.content
    return "".join(b.get("text", "") for b in message.content if isinstance(b, dict))


AGENT_PROMPT = """You are a careful finance assistant for product managers.

Reply with ONLY a JSON object in one of these two forms.

If the question can be answered:
{{"status": "answer",
  "assumptions": ["any interpretation you had to choose, or an empty list"],
  "inputs": [{{"name": "snake_case_name", "value": <number>, "source": "<exact phrase copied from the question>"}}],
  "code": "<Python that uses ONLY the input names as variables and prints the final number>",
  "explanation": "<one sentence>"}}

If answering would require guessing missing data, or the question is ambiguous in a way that changes the answer:
{{"status": "clarify", "clarifying_question": "<what you need to know>"}}

Rules:
- Every number you use must be an input whose "source" is copied word for word from the question.
  Percentages are decimals (8% -> 0.08). Derive other values in the code, not in the inputs.
- The code may not contain other numbers except 0, 1, 2, 12 and 100. No imports. Use print(round(answer, 2)).
- Report percentages as percent numbers (15% -> 15).
- If a reasonable standard interpretation exists, answer and state it in "assumptions".

Question: {question}"""

DIRECT_PROMPT = """Answer this finance question with ONLY the final number (percentages as percent numbers, e.g. 15 for 15%).
If it cannot be answered without more information, reply exactly: CLARIFY: <your question>

Question: {question}"""


def ask(question: str, model, max_attempts: int = 3) -> dict:
  """The agent: structured reply -> input checks -> sandboxed code. Errors go back to the model to fix."""

  prompt, last_error = AGENT_PROMPT.format(question=question), ""
  for attempt in range(1, max_attempts + 1):
    raw = text_of(model.invoke(prompt))
    try:
      reply = parse_reply(raw)
    except (ValidationError, ValueError) as err:
      problems = [f"invalid JSON reply: {err}"]
    else:
      if reply.status == "clarify":
        return {"status": "clarify", "clarifying_question": reply.clarifying_question, "attempts": attempt}
      problems = check_inputs(reply.inputs, question)
      if not problems:
        run = run_sandboxed(reply.code, {i.name: i.value for i in reply.inputs})
        if run["ok"] and first_number(run["output"]) is not None:
          return {"status": "answer", "value": first_number(run["output"]), "output": run["output"], "inputs": reply.inputs, "assumptions": reply.assumptions, "code": reply.code,"explanation": reply.explanation, "attempts": attempt}
        problems = [run["error"] or "the code printed no number"]
    last_error = "; ".join(problems)
    prompt = (f"{AGENT_PROMPT.format(question=question)}\n\nYour previous reply:\n{raw}\n\n"
              f"It was rejected: {last_error}\nFix it and reply with ONLY the JSON object.")
  return {"status": "failed", "error": last_error, "attempts": max_attempts}


def ask_direct(question: str, model) -> dict:
  """The baseline: the LLM answers in one step, with no code and no checks."""
  raw = text_of(model.invoke(DIRECT_PROMPT.format(question=question))).strip()
  if raw.upper().startswith("CLARIFY"):
    return {"status": "clarify", "clarifying_question": raw.split(":", 1)[-1].strip()}
  value = first_number(raw)
  return {"status": "answer" if value is not None else "failed", "value": value, "assumptions": [], "output": raw}


def score(case: dict, result: dict) -> bool:
  """answer: the right number · clarify: asked for more information · clarify_or_assume: asked, or answered with a stated assumption."""
  if case["expect"] == "answer":
    if result["status"] != "answer" or result.get("value") is None:
      return False
    v, target = result["value"], case["value"]
    candidates = [v, v * 100] if case.get("unit") == "percent" else [v]
    return any(abs(c - target) <= max(0.01, abs(target) * 0.001) for c in candidates)
  if case["expect"] == "clarify":
    return result["status"] == "clarify"
  return result["status"] == "clarify" or (result["status"] == "answer" and len(result.get("assumptions", [])) > 0)

## Step 4: Offline tests (no API key)

A scripted model stands in for the LLM, so the input checks, sandbox rules, retry loop and scoring are all tested without any API calls.

In [ ]:
class Scripted:
  """Returns pre-written replies in order: an offline stand-in for an LLM."""

  def __init__(self, replies):
    self.replies, self.prompts = list(replies), []
  def invoke(self, prompt):
    self.prompts.append(prompt)
    return type("Msg", (), {"content": self.replies[min(len(self.prompts) - 1, len(self.replies) - 1)]})()

Q = "If we grow 8% monthly, what is our user count after 6 months starting from 10,000?"
GOOD = json.dumps({"status": "answer", "assumptions": [], "explanation": "Compounded monthly.",
    "inputs": [{"name": "start", "value": 10000, "source": "starting from 10,000"},
               {"name": "rate", "value": 0.08, "source": "8% monthly"},
              {"name": "months", "value": 6, "source": "6 months"}],
    "code": "users = start * (1 + rate) ** months\nprint(round(users, 2))"})

# 1. Input provenance
I = lambda name, value, source: Input(name=name, value=value, source=source)
assert check_inputs([I("rate", 0.08, "8% monthly"), I("rate2", 8, "8% monthly")], Q) == []          # 8% may be 0.08 or 8
assert check_inputs([I("rev", 1_200_000, "$1.2M")], "Revenue grew from $1.2M to $1.5M") == []         # suffixes understood
assert "does not match" in check_inputs([I("rate", 0.8, "8% monthly")], Q)[0]                         # misread value caught
assert "does not appear" in check_inputs([I("cost", 500, "costs of 500")], Q)[0]                      # invented source caught
print("✓ 1. input checks: valid readings accepted; misread values and invented sources rejected")

# 2. Sandbox rules
assert run_sandboxed("print(round(start * (1 + rate) ** months, 2))", {"start": 10000, "rate": 0.08, "months": 6})["output"] == "15868.74"
assert "hard-coded number 10000" in run_sandboxed("print(10000 * 1.08)", {})["error"]                  # bypassing the inputs is blocked
assert "unknown variable" in run_sandboxed("print(round(price * 2, 2))", {})["error"]
assert not run_sandboxed("import os\nprint(1)", {})["ok"]
assert not run_sandboxed("print(().__class__)", {})["ok"]
print("✓ 2. sandbox: injected inputs work; hard-coded numbers, unknown names, imports and attribute tricks blocked")

# 3. The agent loop: an invented number is rejected and fixed on the second attempt
BAD = json.loads(GOOD); BAD["inputs"][1] = {"name": "rate", "value": 0.8, "source": "8% monthly"}
model = Scripted([json.dumps(BAD), GOOD])
result = ask(Q, model)
assert result["status"] == "answer" and result["value"] == 15868.74 and result["attempts"] == 2
assert "does not match any number" in model.prompts[1]                                                 # the model was told why
print("✓ 3. agent: a misread input is sent back with the reason, then corrected")

# 4. Clarification and failure paths
assert ask("What's our break-even point?", Scripted(['{"status": "clarify", "clarifying_question": "What are your costs?"}']))["status"] == "clarify"
assert ask(Q, Scripted(["not json"]))["status"] == "failed"
print("✓ 4. agent: clarifies when told to; gives up cleanly after repeated invalid replies")

# 5. Scoring
assert score({"expect": "answer", "value": 15}, {"status": "answer", "value": 15.0})
assert score({"expect": "answer", "value": 15, "unit": "percent"}, {"status": "answer", "value": 0.15})
assert not score({"expect": "answer", "value": 15}, {"status": "answer", "value": 25.0})
assert score({"expect": "clarify"}, {"status": "clarify"}) and not score({"expect": "clarify"}, {"status": "answer", "value": 1})
assert score({"expect": "clarify_or_assume"}, {"status": "answer", "value": 1, "assumptions": ["10% per month"]})
assert not score({"expect": "clarify_or_assume"}, {"status": "answer", "value": 1, "assumptions": []})
print("✓ 5. scoring: answers, clarifications and stated assumptions graded correctly")
print("\nAll offline tests passed.")

✓ 1. input checks: valid readings accepted; misread values and invented sources rejected
✓ 2. sandbox: injected inputs work; hard-coded numbers, unknown names, imports and attribute tricks blocked
✓ 3. agent: a misread input is sent back with the reason, then corrected
✓ 4. agent: clarifies when told to; gives up cleanly after repeated invalid replies
✓ 5. scoring: answers, clarifications and stated assumptions graded correctly

All offline tests passed.


## Step 5: Connect to the LLM
Set `PROVIDER` to `"openai"` or `"anthropic"`, matching the key in your Secrets.

In [ ]:
import os

def _load(key):
  """Colab Secrets first, then an environment variable."""
  try:
    from google.colab import userdata
    value = userdata.get(key)
    if value:
      return value
  except Exception:
    pass
  return os.environ.get(key)

PROVIDER = "openai"   # or "anthropic"

if PROVIDER == "openai":
  from langchain_openai import ChatOpenAI
  llm = ChatOpenAI(model="gpt-5-mini", api_key=_load("OPENAI_API_KEY"))
elif PROVIDER == "anthropic":
  from langchain_anthropic import ChatAnthropic
  llm = ChatAnthropic(model="claude-haiku-5-5", api_key=_load("ANTHROPIC_API_KEY"), max_tokens=1500)
else:
  raise ValueError('PROVIDER must be "openai" or "anthropic"')

print(f"Using {PROVIDER}")

Using openai


## Step 6: The evaluation set

| Type | Count | Expected behaviour |
|---|---|---|
| `answer` | 13 | The right number. Includes traps: stacked discounts aren't additive, margin isn't markup, monthly compounding |
| `clarify_or_assume` | 2 | Ambiguous wording: ask, or answer with the interpretation stated |
| `clarify` | 3 | Missing data: ask for it instead of inventing numbers |

Expected values are computed in Python below, not typed in by hand.

In [ ]:
EVAL_SET = [
  {"q": "If we grow 8% monthly, what is our user count after 6 months starting from 10,000?", "expect": "answer", "value": 10_000 * 1.08 ** 6},
  {"q": "We lose 25% of our 40,000 users. How many do we retain?", "expect": "answer", "value": 40_000 * 0.75},
  {"q": "If each user pays $10/month and we have 5,000 users, what is the monthly revenue?", "expect": "answer", "value": 10 * 5_000},
  {"q": "If revenue is $50,000 and cost is $37,000, what's our profit?", "expect": "answer", "value": 50_000 - 37_000},
  {"q": "Our customer acquisition cost is $120 and each customer brings $15 of profit per month. How many months until we break even?", "expect": "answer", "value": 120 / 15},
  {"q": "We raised our price from $40 to $46. What is the percentage increase?", "expect": "answer", "value": (46 - 40) / 40 * 100, "unit": "percent"},
  {"q": "We have 20,000 customers and lose 5% of them every month. How many are left after 12 months?", "expect": "answer", "value": 20_000 * 0.95 ** 12},
  {"q": "Revenue grew from $1.2M to $1.5M. What was the percentage growth?", "expect": "answer", "value": (1.5 - 1.2) / 1.2 * 100, "unit": "percent"},
  {"q": "3% of our 80,000 monthly visitors sign up, and each sign-up pays $25. What monthly revenue do sign-ups bring in?", "expect": "answer", "value": 80_000 * 0.03 * 25},
  {"q": "A $200 subscription gets a 15% discount, then a further 10% off the discounted price. What is the final price?", "expect": "answer", "value": 200 * 0.85 * 0.90},
  {"q": "A product costs us $60 and sells for $80. What is the gross margin percentage?", "expect": "answer", "value": (80 - 60) / 80 * 100, "unit": "percent"},
  {"q": "A customer pays $30 per month and stays for 18 months on average. What is their lifetime revenue?", "expect": "answer", "value": 30 * 18},
  {"q": "We put $10,000 in an account paying 5% annual interest, compounded monthly. What is it worth after 3 years?", "expect": "answer", "value": 10_000 * (1 + 0.05 / 12) ** 36},
  {"q": "Our users grew 10% for 3 months from 5,000. How many users do we have now?", "expect": "clarify_or_assume"},
  {"q": "If we cut churn by 2%, how many more of our 50,000 users do we keep each month?", "expect": "clarify_or_assume"},
  {"q": "What's our break-even point in months?", "expect": "clarify"},
  {"q": "How much revenue will we make next year?", "expect": "clarify"},
  {"q": "If we double our marketing budget, how many new customers will we get?", "expect": "clarify"},
]
print(f"{len(EVAL_SET)} cases: " + ", ".join(f"{t}={sum(c['expect'] == t for c in EVAL_SET)}" for t in ["answer", "clarify_or_assume", "clarify"]))

18 cases: answer=13, clarify_or_assume=2, clarify=3


## Step 7: Run both approaches and compare

Each question is answered by the **direct** baseline (one LLM call, no code) and by the **agent** (verified inputs and sandboxed code).

In [ ]:
rows = []
for i, case in enumerate(EVAL_SET, 1):
  direct = ask_direct(case["q"], llm)
  agent = ask(case["q"], llm)
  rows.append({"i": i, "case": case, "direct": direct, "agent": agent, "direct_ok": score(case, direct), "agent_ok": score(case, agent)})
  print(f"{i:>2}. direct {'✓' if rows[-1]['direct_ok'] else '✗'}  agent {'✓' if rows[-1]['agent_ok'] else '✗'}  {case['q'][:70]}")

 1. direct ✓  agent ✓  If we grow 8% monthly, what is our user count after 6 months starting 
 2. direct ✓  agent ✓  We lose 25% of our 40,000 users. How many do we retain?
 3. direct ✓  agent ✓  If each user pays $10/month and we have 5,000 users, what is the month
 4. direct ✓  agent ✓  If revenue is $50,000 and cost is $37,000, what's our profit?
 5. direct ✓  agent ✓  Our customer acquisition cost is $120 and each customer brings $15 of 
 6. direct ✓  agent ✓  We raised our price from $40 to $46. What is the percentage increase?
 7. direct ✓  agent ✓  We have 20,000 customers and lose 5% of them every month. How many are
 8. direct ✓  agent ✓  Revenue grew from $1.2M to $1.5M. What was the percentage growth?
 9. direct ✓  agent ✓  3% of our 80,000 monthly visitors sign up, and each sign-up pays $25. 
10. direct ✓  agent ✓  A $200 subscription gets a 15% discount, then a further 10% off the di
11. direct ✓  agent ✓  A product costs us $60 and sells for $80. What is the gross margin 

## Step 8: Scorecard

In [ ]:
def tally(kind, key):
  group = [r for r in rows if r["case"]["expect"] == kind]
  return f"{sum(r[key] for r in group)}/{len(group)}"

print(f"{'':<34}{'Direct LLM':>12}{'Agent':>10}")
print(f"{'Correct numeric answers':<34}{tally('answer', 'direct_ok'):>12}{tally('answer', 'agent_ok'):>10}")
print(f"{'Ambiguous: asked or stated assumption':<34}{tally('clarify_or_assume', 'direct_ok'):>12}{tally('clarify_or_assume', 'agent_ok'):>10}")
print(f"{'Missing data: asked instead of guessing':<34}{tally('clarify', 'direct_ok'):>12}{tally('clarify', 'agent_ok'):>10}")
print(f"{'Overall':<34}{sum(r['direct_ok'] for r in rows):>9}/{len(rows)}{sum(r['agent_ok'] for r in rows):>7}/{len(rows)}")

retried = [r for r in rows if r["agent"].get("attempts", 1) > 1]
print(f"\nAgent replies that were rejected by the checks and then corrected: {len(retried)}")

print("\nDetails where the two approaches differ, or the agent failed:")
for r in rows:
  if r["direct_ok"] != r["agent_ok"] or not r["agent_ok"]:
    c, d, a = r["case"], r["direct"], r["agent"]
    print("-" * 78)
    print(f"{r['i']}. {c['q']}")
    print(f"   expected: {round(c['value'], 2) if 'value' in c else c['expect']}")
    print(f"   direct:   {d.get('value') if d['status'] == 'answer' else d['status']}  {'✓' if r['direct_ok'] else '✗'}")
    print(f"   agent:    {a.get('value') if a['status'] == 'answer' else a['status']}  {'✓' if r['agent_ok'] else '✗'}"
          + (f"  (assumed: {'; '.join(a['assumptions'])})" if a.get("assumptions") else "")
          + (f"  (asked: {a['clarifying_question']})" if a["status"] == "clarify" else "")
          + (f"  (error: {a['error']})" if a["status"] == "failed" else ""))

                                    Direct LLM     Agent
Correct numeric answers                  13/13     13/13
Ambiguous: asked or stated assumption         0/2       2/2
Missing data: asked instead of guessing         3/3       3/3
Overall                                  16/18     18/18

Agent replies that were rejected by the checks and then corrected: 1

Details where the two approaches differ, or the agent failed:
------------------------------------------------------------------------------
14. Our users grew 10% for 3 months from 5,000. How many users do we have now?
   expected: clarify_or_assume
   direct:   6655.0  ✗
   agent:    6655.0  ✓  (assumed: Interpreting "10%" as a monthly compounded growth rate applied each month for the "3 months" starting from the "5,000" users.)
------------------------------------------------------------------------------
15. If we cut churn by 2%, how many more of our 50,000 users do we keep each month?
   expected: clarify_or_assume
   dire

## Step 9: Look inside one answer

Every input is traceable to the words of the question, which is what makes the answer auditable.

In [ ]:
example = ask("A $200 subscription gets a 15% discount, then a further 10% off the discounted price. What is the final price?", llm)
if example["status"] == "answer":
  print("Inputs (value  ←  source phrase):")
  for inp in example["inputs"]:
    print(f"  {inp.name:<18} {inp.value:<10} ←  \"{inp.source}\"")
  print("Assumptions:", example["assumptions"] or "none")
  print("Code:\n" + "\n".join("  " + line for line in example["code"].splitlines()))
  print("Result:", example["output"])
  print("Explanation:", example["explanation"])
else:
  print(example)

## Notes

- **Reading the comparison honestly:** strong models often get simple arithmetic right on their own, so the direct baseline may score well on easy questions. The agent's advantages show up on multi-step questions, on knowing when *not* to answer, and on **auditability**: every number in an answer can be traced to the question.
- **Why inputs are injected rather than written into the code:** if the code could contain its own numbers, the input check could be bypassed. Rejecting hard-coded values forces every figure through the check.
- **Limits:** the input check confirms a number came from the question, not that the formula is right; that's what the scorecard measures. The sandbox is a demo-level safeguard; production systems run generated code in an isolated container.